# Tutorial 1: Correct part of a spectrum

This tutorial removes telluric absorption from a narrow, reduced
HARPS spectrum around the stellar Na D doublet.

It first runs PyMolFit with only the essential input information.
After inspecting that baseline, it introduces fit and exclusion
regions, displays them on the original spectrum, and repeats the
correction.

## Install

Install PyMolFit and interactive plotting support in the
environment used by this notebook:

```bash
python -m pip install pymolfit ipympl
```

The first correction may download and verify the managed AER
catalogue.

In [ ]:
%matplotlib widget

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pymolfit import correct, load_spectrum

In [ ]:
candidates = (Path.cwd() / "tutorials", Path.cwd())
TUTORIAL_ROOT = next(
    (path.resolve() for path in candidates if (path / "data").is_dir()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the PyMolFit repository or tutorials directory"
    )

INPUT = TUTORIAL_ROOT / "data" / "harps_nad_crop_air.fits"
spectrum = load_spectrum(
    INPUT,
    wavelength_medium="air",
).to_air().to_unit("angstrom")

print(f"Pixels: {spectrum.wavelength.size:,}")
print(
    f"Wavelength range: {spectrum.wavelength[0]:.1f}-"
    f"{spectrum.wavelength[-1]:.1f} Angstrom"
)

## Inspect the original spectrum

Always inspect the complete available input before fitting. No
plotting stride is used: every pixel is displayed.

In [ ]:
valid = spectrum.valid & np.isfinite(spectrum.flux)
scale = np.nanmedian(spectrum.flux[valid])

plt.figure(figsize=(12, 4))
plt.plot(
    spectrum.wavelength[valid],
    spectrum.flux[valid] / scale,
    color="black",
    linewidth=0.7,
)
plt.xlabel("Air wavelength [Angstrom]")
plt.ylabel("Flux / median")
plt.title("Original HARPS Na D interval")
plt.tight_layout()
plt.show()

## First correction: automatic baseline

The first call contains only the file and its wavelength medium.
PyMolFit obtains the line catalogue and atmosphere and chooses the
continuum, instrumental broadening, segmentation, and wavelength
alignment automatically.

In [ ]:
baseline_result = correct(
    input_path=INPUT,
    wavelength_medium="air",
)

if not baseline_result.success:
    raise RuntimeError(baseline_result.message)

## Inspect the baseline

This result lets us see whether strong astrophysical features
influenced the atmospheric parameter estimate before manually
defining any masks.

In [ ]:
baseline_observed = baseline_result.spectrum.to_air().to_unit("angstrom")
baseline_corrected = baseline_result.corrected.to_air().to_unit("angstrom")
scale = np.nanmedian(
    baseline_observed.flux[
        baseline_observed.valid & np.isfinite(baseline_observed.flux)
    ]
)

figure, axes = plt.subplots(
    2,
    1,
    figsize=(12, 7),
    sharex=True,
    height_ratios=(2, 1),
    constrained_layout=True,
)
axes[0].plot(
    baseline_observed.wavelength,
    baseline_observed.flux / scale,
    color="black",
    linewidth=0.7,
    alpha=0.65,
    label="Observed",
)
axes[0].plot(
    baseline_corrected.wavelength,
    baseline_corrected.flux / scale,
    color="tab:blue",
    linewidth=0.8,
    label="Baseline corrected",
)
axes[0].set_ylabel("Flux / median")
axes[0].legend()

axes[1].plot(
    baseline_observed.wavelength,
    baseline_result.transmission,
    color="tab:red",
    linewidth=0.8,
)
axes[1].set_xlabel("Air wavelength [Angstrom]")
axes[1].set_ylabel("Transmission")
axes[1].set_ylim(0.85, 1.01)

figure.suptitle("Automatic baseline correction")
plt.show()

## Choose and inspect fit regions

`FIT_RANGES` selects pixels that may constrain the atmospheric and
instrumental parameters. `EXCLUDE_RANGES` removes the broad
astrophysical Na D absorption from that parameter estimate.

The ranges use the same unshifted air-wavelength frame as this
input and are expressed in microns. Excluded pixels are still
assigned a transmission model and corrected afterward.

In [ ]:
FIT_RANGES = (
    (0.5883, 0.5907),
)

EXCLUDE_RANGES = (
    (0.58875, 0.58996),
)

plt.figure(figsize=(12, 4))
plt.plot(
    spectrum.wavelength[valid],
    spectrum.flux[valid] / scale,
    color="black",
    linewidth=0.7,
)
for index, (lower, upper) in enumerate(FIT_RANGES):
    plt.axvspan(
        lower * 1e4,
        upper * 1e4,
        color="tab:green",
        alpha=0.10,
        label="Fit range" if index == 0 else None,
    )
for index, (lower, upper) in enumerate(EXCLUDE_RANGES):
    plt.axvspan(
        lower * 1e4,
        upper * 1e4,
        color="tab:red",
        alpha=0.14,
        label="Excluded from fit" if index == 0 else None,
    )
plt.xlabel("Air wavelength [Angstrom]")
plt.ylabel("Flux / median")
plt.title("Fit and exclusion regions on the original spectrum")
plt.legend()
plt.tight_layout()
plt.show()

## Second correction: use the inspected regions

Only the two scientifically selected masks are added to the
essential baseline call. All other behavior remains automatic.

In [ ]:
refined_result = correct(
    input_path=INPUT,
    wavelength_medium="air",
    fit_ranges=FIT_RANGES,
    exclude_ranges=EXCLUDE_RANGES,
)

if not refined_result.success:
    raise RuntimeError(refined_result.message)

## Compare both corrections

The astrophysical Na D lines should remain present. The masks only
control parameter estimation; they do not remove those pixels from
the returned spectrum.

In [ ]:
refined_corrected = refined_result.corrected.to_air().to_unit("angstrom")

figure, axes = plt.subplots(
    2,
    1,
    figsize=(12, 7),
    sharex=True,
    height_ratios=(2, 1),
    constrained_layout=True,
)
axes[0].plot(
    baseline_observed.wavelength,
    baseline_observed.flux / scale,
    color="black",
    linewidth=0.7,
    alpha=0.55,
    label="Observed",
)
axes[0].plot(
    baseline_corrected.wavelength,
    baseline_corrected.flux / scale,
    color="tab:orange",
    linewidth=0.8,
    label="Automatic baseline",
)
axes[0].plot(
    refined_corrected.wavelength,
    refined_corrected.flux / scale,
    color="tab:blue",
    linewidth=0.8,
    label="Using fit masks",
)
axes[0].set_ylabel("Flux / median")
axes[0].legend()

axes[1].plot(
    baseline_observed.wavelength,
    baseline_result.transmission,
    color="tab:orange",
    linewidth=0.8,
    label="Baseline transmission",
)
axes[1].plot(
    baseline_observed.wavelength,
    refined_result.transmission,
    color="tab:blue",
    linewidth=0.8,
    label="Masked-fit transmission",
)
axes[1].set_xlabel("Air wavelength [Angstrom]")
axes[1].set_ylabel("Transmission")
axes[1].set_ylim(0.85, 1.01)
axes[1].legend()

figure.suptitle("Partial-spectrum telluric correction")
plt.show()

## Saving is optional

Correction and saving are separate operations. Save only the
corrected spectrum with
`save_corrected_txt(refined_result, "corrected.txt")`, or retain
the complete auditable result with
`save_fit_product_ecsv(refined_result, "fit_product.ecsv")`.